# Integración de 2 variables al dataset: Health Percentage y GDP

## Objetivo

Construir el dataset solicitado en la actividad integrando las variables:

- `percentage_health`
- `gdp`

El resultado final tendrá exactamente estas 8 columnas:

`entity`, `year`, `population`, `births`, `deaths`, `child_deaths_under_5`, `percentage_health`, `gdp`

Los datos se integran mediante `entity` y `year`.


In [1]:
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)


## 1. Cargar el dataset demográfico base

Se utiliza el dataset de Our World in Data que contiene nacimientos y defunciones, como en el notebook trabajado anteriormente.

Después se conservan los años de 2000 a 2023, que corresponden al periodo de los indicadores solicitados.


In [2]:
url_births_deaths = "https://ourworldindata.org/grapher/births-and-deaths-projected-to-2100.csv?v=1&csvType=full&useColumnShortNames=true"

df_base = pd.read_csv(
    url_births_deaths,
    storage_options={"User-Agent": "Our World In Data data fetch/1.0"}
)

print("Columnas originales:")
print(df_base.columns.tolist())
print("Dimensiones:", df_base.shape)


Columnas originales:
['entity', 'code', 'year', 'deaths__sex_all__age_all__variant_estimates', 'deaths__sex_all__age_all__variant_medium__projected', 'births__sex_all__age_all__variant_estimates', 'births__sex_all__age_all__variant_medium__projected']
Dimensiones: (39562, 7)


In [3]:
# Renombrar las columnas del dataset demográfico
df_base = df_base.rename(columns={
    "entity": "entity",
    "year": "year",
    "births__sex_all__age_all__variant_estimates": "births_estimates",
    "births__sex_all__age_all__variant_medium__projected": "births_projected",
    "deaths__sex_all__age_all__variant_estimates": "deaths_estimates",
    "deaths__sex_all__age_all__variant_medium__projected": "deaths_projected"
})

# Combinar estimaciones y proyecciones
df_base["births"] = df_base["births_estimates"].fillna(df_base["births_projected"])
df_base["deaths"] = df_base["deaths_estimates"].fillna(df_base["deaths_projected"])

# Solo necesitamos 2000-2023
df_base = df_base[
    (df_base["year"] >= 2000) &
    (df_base["year"] <= 2023)
].copy()

df_base = df_base[["entity", "year", "births", "deaths"]]

print("Dataset base:")
display(df_base.head())
print("Dimensiones:", df_base.shape)


Dataset base:


,entity,year,births,deaths
50,Afghanistan,2000,1036525.0,253804.0
51,Afghanistan,2001,1006562.0,242473.0
52,Afghanistan,2002,1015202.0,239161.0
53,Afghanistan,2003,1069727.0,243573.0
54,Afghanistan,2004,1095372.0,244435.0


Dimensiones: (6288, 4)


## 2. Agregar population

Se utiliza el indicador de población de Our World in Data y se integra mediante `entity` y `year`.


In [4]:
url_population = "https://ourworldindata.org/grapher/population-unwpp.csv?v=1&csvType=full&useColumnShortNames=true"

df_population = pd.read_csv(
    url_population,
    storage_options={"User-Agent": "Our World In Data data fetch/1.0"}
)

print("Columnas population:")
print(df_population.columns.tolist())

# Detectar automáticamente la columna del indicador
population_col = [
    c for c in df_population.columns
    if c not in ["entity", "code", "year"]
][0]

df_population = df_population.rename(columns={
    population_col: "population"
})

df_population = df_population[
    (df_population["year"] >= 2000) &
    (df_population["year"] <= 2023)
][["entity", "year", "population"]]

df = df_base.merge(
    df_population,
    on=["entity", "year"],
    how="left"
)

print("Después de population:", df.shape)
display(df.head())


Columnas population:
['entity', 'code', 'year', 'population__sex_all__age_all__variant_estimates']
Después de population: (6288, 5)


,entity,year,births,deaths,population
0,Afghanistan,2000,1036525.0,253804.0,20130334
1,Afghanistan,2001,1006562.0,242473.0,20284303
2,Afghanistan,2002,1015202.0,239161.0,21378123
3,Afghanistan,2003,1069727.0,243573.0,22733053
4,Afghanistan,2004,1095372.0,244435.0,23560656


## 3. Agregar child_deaths_under_5

El indicador corresponde al número de niños que mueren antes de cumplir los 5 años.


In [5]:
url_child_deaths = "https://ourworldindata.org/grapher/number-of-child-deaths-unwpp.csv?v=1&csvType=full&useColumnShortNames=true"

df_child_deaths = pd.read_csv(
    url_child_deaths,
    storage_options={"User-Agent": "Our World In Data data fetch/1.0"}
)

print("Columnas child deaths:")
print(df_child_deaths.columns.tolist())

child_deaths_col = [
    c for c in df_child_deaths.columns
    if c not in ["entity", "code", "year"]
][0]

df_child_deaths = df_child_deaths.rename(columns={
    child_deaths_col: "child_deaths_under_5"
})

df_child_deaths = df_child_deaths[
    (df_child_deaths["year"] >= 2000) &
    (df_child_deaths["year"] <= 2023)
][["entity", "year", "child_deaths_under_5"]]

df = df.merge(
    df_child_deaths,
    on=["entity", "year"],
    how="left"
)

print("Después de child deaths:", df.shape)
display(df.head())


Columnas child deaths:
['entity', 'code', 'year', 'deaths__sex_all__age_0_4__variant_estimates']
Después de child deaths: (6288, 6)


,entity,year,births,deaths,population,child_deaths_under_5
0,Afghanistan,2000,1036525.0,253804.0,20130334,133112.0
1,Afghanistan,2001,1006562.0,242473.0,20284303,125627.0
2,Afghanistan,2002,1015202.0,239161.0,21378123,123215.0
3,Afghanistan,2003,1069727.0,243573.0,22733053,123894.0
4,Afghanistan,2004,1095372.0,244435.0,23560656,122315.0


## 4. Agregar percentage_health

Dataset solicitado en la actividad:

**Total healthcare spending as a share of GDP, 2000-2023.**


In [6]:
url_health = "https://ourworldindata.org/grapher/total-healthcare-expenditure-gdp.csv?v=1&csvType=full&useColumnShortNames=true"

df_health = pd.read_csv(
    url_health,
    storage_options={"User-Agent": "Our World In Data data fetch/1.0"}
)

print("Columnas health:")
print(df_health.columns.tolist())

health_col = [
    c for c in df_health.columns
    if c not in ["entity", "code", "year"]
][0]

df_health = df_health.rename(columns={
    health_col: "percentage_health"
})

df_health = df_health[
    (df_health["year"] >= 2000) &
    (df_health["year"] <= 2023)
][["entity", "year", "percentage_health"]]

df = df.merge(
    df_health,
    on=["entity", "year"],
    how="left"
)

print("Después de percentage_health:", df.shape)
display(df.head())


Columnas health:
['entity', 'code', 'year', 'current_health_expenditure__che__as_percentage_of_gross_domestic_product__gdp__pct']
Después de percentage_health: (6288, 7)


,entity,year,births,deaths,population,child_deaths_under_5,percentage_health
0,Afghanistan,2000,1036525.0,253804.0,20130334,133112.0,NaN
1,Afghanistan,2001,1006562.0,242473.0,20284303,125627.0,NaN
2,Afghanistan,2002,1015202.0,239161.0,21378123,123215.0,9.443391
3,Afghanistan,2003,1069727.0,243573.0,22733053,123894.0,8.941258
4,Afghanistan,2004,1095372.0,244435.0,23560656,122315.0,9.808474


## 5. Agregar GDP per capita

Se utiliza el indicador **GDP per capita** proporcionado en la actividad.


In [8]:
url_gdp = "https://ourworldindata.org/grapher/gdp-per-capita-worldbank.csv?v=1&csvType=full&useColumnShortName=true"

df_gdp = pd.read_csv(
    url_gdp,
    storage_options={"User-Agent": "Our World In Data data fetch/1.0"}
)

print("Columnas originales de GDP:")
print(df_gdp.columns.tolist())

# Normalizar nombres
df_gdp = df_gdp.rename(columns={
    "Entity": "entity",
    "Code": "code",
    "Year": "year"
})

print("\nColumnas después de renombrar:")
print(df_gdp.columns.tolist())

# Buscar la columna del indicador GDP
gdp_col = [
    c for c in df_gdp.columns
    if c not in ["entity", "code", "year"]
][0]

print("\nColumna utilizada como GDP:", gdp_col)

# Renombrar el indicador
df_gdp = df_gdp.rename(columns={
    gdp_col: "gdp"
})

# Asegurar que year sea numérico
df_gdp["year"] = pd.to_numeric(
    df_gdp["year"],
    errors="coerce"
)

# Filtrar periodo solicitado
df_gdp = df_gdp[
    (df_gdp["year"] >= 2000) &
    (df_gdp["year"] <= 2023)
][["entity", "year", "gdp"]]

print("\nDataset GDP:")
print(df_gdp.shape)

display(df_gdp.head())

Columnas originales de GDP:
['Entity', 'Code', 'Year', 'GDP per capita', 'World region according to OWID']

Columnas después de renombrar:
['entity', 'code', 'year', 'GDP per capita', 'World region according to OWID']

Columna utilizada como GDP: GDP per capita

Dataset GDP:
(5033, 3)


,entity,year,gdp
0,Afghanistan,2000,1617.8264
1,Afghanistan,2001,1454.1108
2,Afghanistan,2002,1774.3087
3,Afghanistan,2003,1815.9282
4,Afghanistan,2004,1776.9182


## 6. Seleccionar las 8 columnas solicitadas

Finalmente se ordenan las columnas exactamente como las solicita la actividad.


In [10]:
# ============================================================
# INTEGRAR GDP AL DATASET
# ============================================================

url_gdp = (
    "https://ourworldindata.org/grapher/"
    "gdp-per-capita-worldbank.csv"
    "?v=1&csvType=full&useColumnShortNames=true"
)

storage_options = {
    "User-Agent": "Our World In Data data fetch/1.0"
}

# Descargar GDP
df_gdp = pd.read_csv(
    url_gdp,
    storage_options=storage_options
)

print("Columnas originales de GDP:")
print(df_gdp.columns.tolist())


# ============================================================
# NORMALIZAR Entity / Year / Code
# ============================================================

renombrar = {}

for col in df_gdp.columns:

    if str(col).strip().lower() == "entity":
        renombrar[col] = "entity"

    elif str(col).strip().lower() == "year":
        renombrar[col] = "year"

    elif str(col).strip().lower() == "code":
        renombrar[col] = "code"

df_gdp = df_gdp.rename(columns=renombrar)


print("\nColumnas después de normalizar:")
print(df_gdp.columns.tolist())


# ============================================================
# BUSCAR AUTOMÁTICAMENTE LA COLUMNA DEL GDP
# ============================================================

columnas_gdp = [
    col for col in df_gdp.columns
    if col not in ["entity", "code", "year"]
]

print("\nColumnas de datos encontradas:")
print(columnas_gdp)


if len(columnas_gdp) == 0:
    raise ValueError(
        "No se encontró la columna que contiene el GDP."
    )


# La columna de datos del CSV se convierte en "gdp"
columna_gdp = columnas_gdp[0]

df_gdp = df_gdp.rename(
    columns={
        columna_gdp: "gdp"
    }
)


# ============================================================
# LIMPIAR DATOS
# ============================================================

df_gdp["year"] = pd.to_numeric(
    df_gdp["year"],
    errors="coerce"
)

df_gdp["gdp"] = pd.to_numeric(
    df_gdp["gdp"],
    errors="coerce"
)


# Solo años 2000-2023
df_gdp = df_gdp[
    (df_gdp["year"] >= 2000) &
    (df_gdp["year"] <= 2023)
]


# Dejar solamente las columnas necesarias
df_gdp = df_gdp[
    ["entity", "year", "gdp"]
].copy()


print("\nGDP preparado:")
print(df_gdp.head())


# ============================================================
# INTEGRAR GDP CON EL DATASET PRINCIPAL
# ============================================================

df = df.merge(
    df_gdp,
    on=["entity", "year"],
    how="left"
)


print("\nColumnas de df después de integrar GDP:")
print(df.columns.tolist())


# ============================================================
# VERIFICAR QUE GDP SE INTEGRÓ
# ============================================================

print("\nPrimeros registros con GDP:")

display(
    df[
        ["entity", "year", "gdp"]
    ].head(10)
)


# ============================================================
# COLUMNAS FINALES
# ============================================================

columnas_finales = [
    "entity",
    "year",
    "population",
    "births",
    "deaths",
    "child_deaths_under_5",
    "percentage_health",
    "gdp"
]

df_final = df[columnas_finales].copy()

print("\nColumnas finales:")
print(df_final.columns.tolist())

display(df_final.head(10))

Columnas originales de GDP:
['entity', 'code', 'year', 'ny_gdp_pcap_pp_kd', 'owid_region']

Columnas después de normalizar:
['entity', 'code', 'year', 'ny_gdp_pcap_pp_kd', 'owid_region']

Columnas de datos encontradas:
['ny_gdp_pcap_pp_kd', 'owid_region']

GDP preparado:
        entity  year        gdp
0  Afghanistan  2000  1617.8264
1  Afghanistan  2001  1454.1108
2  Afghanistan  2002  1774.3087
3  Afghanistan  2003  1815.9282
4  Afghanistan  2004  1776.9182

Columnas de df después de integrar GDP:
['entity', 'year', 'births', 'deaths', 'population', 'child_deaths_under_5', 'percentage_health', 'gdp']

Primeros registros con GDP:


,entity,year,gdp
0,Afghanistan,2000,1617.8264
1,Afghanistan,2001,1454.1108
2,Afghanistan,2002,1774.3087
3,Afghanistan,2003,1815.9282
4,Afghanistan,2004,1776.9182
5,Afghanistan,2005,1908.1147
6,Afghanistan,2006,1929.7239
7,Afghanistan,2007,2155.3530
8,Afghanistan,2008,2191.5044
9,Afghanistan,2009,2565.0220



Columnas finales:
['entity', 'year', 'population', 'births', 'deaths', 'child_deaths_under_5', 'percentage_health', 'gdp']


,entity,year,population,births,deaths,child_deaths_under_5,percentage_health,gdp
0,Afghanistan,2000,20130334,1036525.0,253804.0,133112.0,NaN,1617.8264
1,Afghanistan,2001,20284303,1006562.0,242473.0,125627.0,NaN,1454.1108
2,Afghanistan,2002,21378123,1015202.0,239161.0,123215.0,9.443391,1774.3087
3,Afghanistan,2003,22733053,1069727.0,243573.0,123894.0,8.941258,1815.9282
4,Afghanistan,2004,23560656,1095372.0,244435.0,122315.0,9.808474,1776.9182
5,Afghanistan,2005,24404575,1097038.0,241932.0,118420.0,9.948289,1908.1147
6,Afghanistan,2006,25424100,1120880.0,245586.0,116622.0,10.622766,1929.7239
7,Afghanistan,2007,25909852,1135656.0,245809.0,112800.0,9.904675,2155.3530
8,Afghanistan,2008,26482631,1126525.0,235284.0,106511.0,10.256495,2191.5044
9,Afghanistan,2009,27466102,1158017.0,237450.0,106480.0,9.818487,2565.0220


## 7. Verificar el dataset final


In [11]:
print("Número de filas:", len(df_final))
print("Número de columnas:", len(df_final.columns))

print("\nValores faltantes:")
display(df_final.isna().sum().to_frame("valores_faltantes"))

print("\nTipos de datos:")
display(df_final.dtypes.to_frame("tipo"))

print("\nRango de años:")
print(df_final["year"].min(), "a", df_final["year"].max())


Número de filas: 6288
Número de columnas: 8

Valores faltantes:


,valores_faltantes
entity,0
year,0
population,0
births,72
deaths,144
child_deaths_under_5,146
percentage_health,1676
gdp,1447



Tipos de datos:


,tipo
entity,str
year,int64
population,int64
births,float64
deaths,float64
child_deaths_under_5,float64
percentage_health,float64
gdp,float64



Rango de años:
2000 a 2023


## 8. Resultado final

El dataset contiene exactamente las 8 variables solicitadas.


In [12]:
display(df_final.head(20))


,entity,year,population,births,deaths,child_deaths_under_5,percentage_health,gdp
0,Afghanistan,2000,20130334,1036525.0,253804.0,133112.0,NaN,1617.8264
1,Afghanistan,2001,20284303,1006562.0,242473.0,125627.0,NaN,1454.1108
2,Afghanistan,2002,21378123,1015202.0,239161.0,123215.0,9.443391,1774.3087
3,Afghanistan,2003,22733053,1069727.0,243573.0,123894.0,8.941258,1815.9282
4,Afghanistan,2004,23560656,1095372.0,244435.0,122315.0,9.808474,1776.9182
5,Afghanistan,2005,24404575,1097038.0,241932.0,118420.0,9.948289,1908.1147
6,Afghanistan,2006,25424100,1120880.0,245586.0,116622.0,10.622766,1929.7239
7,Afghanistan,2007,25909852,1135656.0,245809.0,112800.0,9.904675,2155.3530
8,Afghanistan,2008,26482631,1126525.0,235284.0,106511.0,10.256495,2191.5044
9,Afghanistan,2009,27466102,1158017.0,237450.0,106480.0,9.818487,2565.0220


## 9. Guardar el dataset final

Se genera un CSV que puede utilizarse para entregar o continuar trabajando.


In [13]:
df_final.to_csv("dataset_final_health_gdp.csv", index=False)

print("Archivo generado: dataset_final_health_gdp.csv")


Archivo generado: dataset_final_health_gdp.csv
